# CAMS CO: просмотр данных и зависимость качества от объёма обучения

Запускайте ячейки последовательно из репозитория. Сначала распаковка и просмотр, затем длительное обучение. Исходные архивы сохраняются; понадобится место под распакованные NetCDF и два массива выбранного региона.

Гипотезы: (1) ошибка AE на фиксированных новых днях уменьшается при увеличении числа обучающих дней; (2) SAM улучшает качество относительно Plain CAE; (3) сравнение AE с PCA и DCT при одинаковой размерности. Результат заранее не предполагается. Один месяц — пилот с малым числом независимых дней, а не доказательство достаточности большого датасета.


In [ ]:
# При необходимости выполните один раз и перезапустите kernel:
# %pip install numpy pandas matplotlib netCDF4 scipy scikit-learn torch
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/models.py').exists()), None)
assert ROOT is not None, 'Откройте ноутбук внутри репозитория DeepCompreesion'
sys.path.insert(0,str(ROOT))
from scripts.cams_experiment import (unpack, inspect_files, experiment_directory, prepare,
                                     split_days, run_experiments)

CONFIG = {
    'data_dir': 'data/cams_co_cross_year',
    'variable': None,  # автоопределение co / co_conc; при необходимости задайте имя
    'center_lat_lon': [51.0, 10.0],  # географический центр домена CAMS Europe
    'crop_shape': [96, 84],         # реальные ячейки, без интерполяции
    'train_day_counts': [7, 30, 90, 180, 365], # полный train pool (731 день) добавляется автоматически
    'split_years': {'train': [2020, 2021], 'validation': 2022, 'test': 2023},
    'evaluation_days_per_month': 7, # из каждого скачанного месяца val/test
    'gap_days': 1,
    'subset_seed': 42,
    'latent_dims': [32, 64],
    'seeds': [42],                 # пилот; для оценки разброса задайте [42, 43, 44] ДО запуска
    'epochs': 100,                 # без early stopping, минимум validation loss
    'batch_size': 8,
    'learning_rate': 0.001,
    'weight_decay': 0.000001,
    'dropout': 0.1,
    'mae_weight': 0.2,
}


## 1. Распаковать и посмотреть содержимое
Архивы извлекаются в `data/cams_co/unpacked/`. Повторный запуск пропускает завершённую распаковку. Число кадров в таблице относится к конкретному файлу: если высоты лежат в разных файлах, складывать эти числа нельзя. Следующая ячейка собирает уникальные timestamps и высоты и выявляет дубли/пропуски.


In [ ]:
files = unpack(ROOT/CONFIG['data_dir'])
inventory, records, reference = inspect_files(files, CONFIG['variable'])
display(inventory)
print('Исходная сетка (lat, lon):',len(reference[0]),len(reference[1]))
print('Единицы CO:',reference[2],'; единицы высоты:',reference[3])
print('Всего уникальных timestamps:',len({t for r in records for t in r['dates']}))
print('Высоты:',sorted({float(z) for r in records for z in r['levels']}))


## 2. Подготовить регион и показать поля
На диск сохраняются массивы `(time, level, latitude, longitude)` в float32. В память читается один пространственный кадр выбранного региона, не весь месячный файл.

Нормализация совпадает по смыслу с предыдущим экспериментом: min–max для каждого кадра и высоты. Это обратимое преобразование, использующее дополнительные `2 × число высот` чисел на кадр; учитываем их в таблице. Физические метрики вычисляются после обратного преобразования в единицах исходного файла. Метрики CAMS нельзя напрямую смешивать с WRF-Chem.


In [ ]:
OUTPUT = experiment_directory(ROOT,CONFIG,files)
inventory.to_csv(OUTPUT/'inventory.csv',index=False)
raw, X, minima, scales, frames, summary = prepare(records,reference,CONFIG,OUTPUT)
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('Результаты:',OUTPUT)
fig, axes = plt.subplots(1,3,figsize=(15,4))
levels_to_show = [0,len(summary['levels'])//2,len(summary['levels'])-1]
for ax,z in zip(axes,levels_to_show):
    im=ax.imshow(raw[0,z],origin='upper',aspect='auto')
    ax.set_title(f"Первый кадр, высота {summary['levels'][z]} {summary['level_units']}")
    ax.set_xlabel('Индекс longitude'); ax.set_ylabel('Индекс latitude')
    fig.colorbar(im,ax=ax,label=summary['units'])
fig.tight_layout(); fig.savefig(OUTPUT/'first_frame.png',dpi=150); plt.show()
plt.figure(figsize=(12,3))
plt.plot(frames.time,np.mean(raw,axis=(1,2,3)))
plt.ylabel(f"Среднее CO, {summary['units']}"); plt.xlabel('Время'); plt.tight_layout()
plt.savefig(OUTPUT/'co_time_series.png',dpi=150); plt.show()


## 3. Зафиксировать разбиение
Последние ~16% дней — test, предшествующие ~16% — validation. Между train/validation и validation/test исключается один календарный день. Из оставшегося train-пула берутся вложенные случайные наборы целых дней. Test/validation одинаковы для всех объёмов и seed; размерности/архитектуры нельзя выбирать по test.

Пространственные патчи из одного timestamp не размножаются между выборками. Соседние дни всё равно могут коррелировать; это пилотное временное разбиение, не независимые атмосферные реализации. PCA fit выполняется только на train.


In [ ]:
splits, counts = split_days(frames,CONFIG,OUTPUT)
display(pd.read_csv(OUTPUT/'partitions.csv').groupby('partition').agg(frames=('frame','size'),days=('day','nunique'),first=('time','min'),last=('time','max')))
display(pd.DataFrame([{'train_days':n,'frames':len(splits[f'train_{n}'])} for n in counts]))
assert min(len(splits[f'train_{n}']) for n in counts) >= max(CONFIG['latent_dims']), 'Увеличьте минимальный train или уменьшите latent_dims для PCA'
print('Число комбинаций:',len(counts)*len(CONFIG['latent_dims'])*len(CONFIG['seeds'])*4)


## 4. Обучение и оценка (долгая ячейка)
Используются существующие `PlainConv3DAutoencoder` и `Conv3DAutoencoder`, размер входа автоматически берётся из CAMS. Каждая сеть проходит все 100 эпох, сохраняется минимум `MSE + 0.2 MAE` на validation. Логи, checkpoint и метрики каждого завершённого запуска записываются сразу. Повторный запуск пропускает завершённые комбинации; незавершённая комбинация обучается заново.

DCT оставляет top-k по модулю коэффициентов отдельно для каждого кадра; это не обучаемый метод. Он хранит значения **и индексы**, поэтому одинаковый k не означает одинаковый размер записи. `payload_bytes` показывает текущий формат реализации без упаковки; отдельно указаны min/max-нормировка и число параметров AE. Размер обученных весов/PCA-базиса не входит в payload. Повторение DCT по объёмам/seed служит горизонтальным ориентиром, а не независимыми измерениями.


In [ ]:
metrics = run_experiments(X,raw,minima,scales,frames,splits,counts,CONFIG,OUTPUT)
display(metrics[metrics.split=='test'][['method','latent_dim','train_days','seed','mse','rmse','relative_l2','ssim','physical_rmse','best_epoch']])


## 5. Кривые для проверки гипотез
Главный результат — test error как функция числа обучающих **дней**, при неизменных test/validation. Параллельно смотрим train/validation: разрыв может указывать на переобучение, а плохие обе ошибки — на ограничения оптимизации или модели. 100 эпох дают больше gradient updates для больших наборов, поэтому эта кривая характеризует фиксированный бюджет эпох, а не изолирует влияние числа данных от числа шагов оптимизации.

CSV содержит pooled MSE/MAE/RMSE, среднюю покадровую relative L2 и средний SSIM по срезам (data_range=1), физические ошибки и покадровые/подневные результаты. При одном seed нет оценки устойчивости к инициализации; стандартное отклонение по трём seed также не является доверительным интервалом по независимым датасетам.


In [ ]:
test = metrics[metrics.split=='test']
columns=['mse','rmse','mae','relative_l2','ssim','physical_rmse']
summary_table=test.groupby(['method','latent_dim','train_days'])[columns].agg(['mean','std','count'])
summary_table.to_csv(OUTPUT/'test_summary.csv'); display(summary_table)
for dim in CONFIG['latent_dims']:
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for ax,metric in zip(axes,['mse','relative_l2','ssim']):
        for name,group in test[test.latent_dim==dim].groupby('method'):
            stat=group.groupby('train_days')[metric].agg(['mean','std']).sort_index()
            ax.plot(stat.index,stat['mean'],marker='o',label=name)
            if stat['std'].notna().any():
                ax.fill_between(stat.index,stat['mean']-stat['std'],stat['mean']+stat['std'],alpha=.15)
        ax.set_xlabel('Обучающих дней'); ax.set_ylabel(metric); ax.grid(alpha=.2)
    axes[0].legend(fontsize=7); fig.suptitle(f'CAMS, latent dimension = {dim}')
    fig.tight_layout(); fig.savefig(OUTPUT/f'learning_curve_dim{dim}.png',dpi=180); plt.show()
print('Для анализа пришлите metrics_all_runs.csv, data_summary.json и partitions.csv из',OUTPUT)
